In [1]:
#Memanggil Library
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# Pengaturan tampilan
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

In [3]:
# =========================================================
# 1. MEMUAT DATASET
# =========================================================
df = pd.read_csv('housing.csv')
print("Dataset berhasil dimuat. Jumlah baris & kolom:", df.shape)

Dataset berhasil dimuat. Jumlah baris & kolom: (20640, 10)


In [4]:
# =========================================================
# 2. MEMERIKSA DATA
# =========================================================
print("\n=== 5 Baris Pertama (.head()) ===")
print(df.head())

print("\n=== Statistik Ringkasan (.describe()) ===")
print(df.describe())

print("\n=== Struktur Data (.info()) ===")
print(df.info())


=== 5 Baris Pertama (.head()) ===
   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1      2401.0      1138.0         8.3014            358500.0        NEAR BAY  
2       496.0       177.0         7.2574            352100.0        NEAR BAY  
3       558.0       219.0         5.6431            341300.0        NEAR BAY  
4       565.0       259.0         3.8462            342200.0        NEAR BAY  

=== Statistik Ringkasan (.describe()) =

In [5]:
# =========================================================
# 3. MENANGANI NILAI YANG HILANG
# =========================================================
print("\n=== Jumlah Nilai yang Hilang per Kolom ===")
print(df.isnull().sum())


=== Jumlah Nilai yang Hilang per Kolom ===
longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64


In [6]:
median_bedrooms = df['total_bedrooms'].median()
df['total_bedrooms'] = df['total_bedrooms'].fillna(median_bedrooms)
print(f"\nNilai kosong pada 'total_bedrooms' diisi dengan median: {median_bedrooms}")
print("\n=== Verifikasi Setelah Imputasi ===")
print(df.isnull().sum())


Nilai kosong pada 'total_bedrooms' diisi dengan median: 435.0

=== Verifikasi Setelah Imputasi ===
longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
ocean_proximity       0
dtype: int64


In [7]:
# =========================================================
# 4. MEMVISUALISASIKAN HUBUNGAN
# =========================================================

# --- Heatmap Korelasi ---
plt.figure(figsize=(10, 8))
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Heatmap Korelasi Antar Fitur Numerik')
plt.tight_layout()
plt.savefig('heatmap_korelasi.png', dpi=150)
plt.close()
print("\nHeatmap korelasi disimpan sebagai 'heatmap_korelasi.png'")

# Korelasi tiap fitur terhadap target (median_house_value)
print("\n=== Korelasi Fitur terhadap median_house_value ===")
print(corr_matrix['median_house_value'].sort_values(ascending=False))

# --- Pairplot untuk fitur-fitur kunci ---
fitur_kunci = ['median_house_value', 'median_income', 'population',
               'total_rooms', 'housing_median_age']
sns.pairplot(df[fitur_kunci], diag_kind='kde', plot_kws={'alpha': 0.3, 's': 10})
plt.savefig('pairplot_fitur_kunci.png', dpi=150)
plt.close()
print("Pairplot fitur kunci disimpan sebagai 'pairplot_fitur_kunci.png'")

# --- Scatter plot: median_income vs median_house_value ---
plt.figure(figsize=(8, 6))
plt.scatter(df['median_income'], df['median_house_value'], alpha=0.2, s=10)
plt.xlabel('Median Income')
plt.ylabel('Median House Value')
plt.title('Hubungan Pendapatan Median vs Harga Rumah')
plt.tight_layout()
plt.savefig('scatter_income_vs_price.png', dpi=150)
plt.close()
print("Scatter plot disimpan sebagai 'scatter_income_vs_price.png'")


Heatmap korelasi disimpan sebagai 'heatmap_korelasi.png'

=== Korelasi Fitur terhadap median_house_value ===
median_house_value    1.000000
median_income         0.688075
total_rooms           0.134153
housing_median_age    0.105623
households            0.065843
total_bedrooms        0.049457
population           -0.024650
longitude            -0.045967
latitude             -0.144160
Name: median_house_value, dtype: float64
Pairplot fitur kunci disimpan sebagai 'pairplot_fitur_kunci.png'
Scatter plot disimpan sebagai 'scatter_income_vs_price.png'


In [10]:
# =========================================================
# 5. REKAYASA FITUR (FEATURE ENGINEERING)
# =========================================================

# a) Rasio kamar per rumah tangga
df['rooms_per_household'] = df['total_rooms'] / df['households']

# b) Rasio kamar tidur per total kamar
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']

# c) Rata-rata jumlah anggota rumah tangga (populasi per rumah tangga)
df['population_per_household'] = df['population'] / df['households']

# d) Menggabungkan latitude & longitude menjadi satu fitur (opsional):
#    Salah satu pendekatan sederhana adalah membuat cluster lokasi
#    menggunakan KMeans, lalu menyimpan label cluster sebagai fitur baru.
from sklearn.cluster import KMeans

koordinat = df[['latitude', 'longitude']]
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
df['location_cluster'] = kmeans.fit_predict(koordinat)

print("\n=== Fitur Baru yang Ditambahkan ===")
print(df[['rooms_per_household', 'bedrooms_per_room',
          'population_per_household', 'location_cluster']].head())

# --- Statistik ringkasan per cluster lokasi ---
# Digunakan untuk menginterpretasikan masing-masing cluster: di mana rata-rata
# lokasinya, berapa harga & income rata-ratanya, dan kategori ocean_proximity
# apa yang paling dominan di cluster tersebut.
cluster_summary = df.groupby('location_cluster').agg(
    jumlah=('median_house_value', 'count'),
    lat_rata=('latitude', 'mean'),
    long_rata=('longitude', 'mean'),
    harga_rata=('median_house_value', 'mean'),
    income_rata=('median_income', 'mean'),
).round(2)

dominan_ocean = df.groupby('location_cluster')['ocean_proximity'] \
                   .agg(lambda x: x.value_counts().idxmax())
cluster_summary['ocean_dominan'] = dominan_ocean
cluster_summary = cluster_summary.sort_values('harga_rata', ascending=False)

print("\n=== Statistik per Cluster Lokasi (diurutkan dari harga tertinggi) ===")
print(cluster_summary.to_string())

cluster_summary.to_csv('cluster_summary.csv')
print("\nRingkasan per cluster disimpan sebagai 'cluster_summary.csv'")

# Cek ulang korelasi fitur baru terhadap target
numeric_df2 = df.select_dtypes(include=[np.number])
print("\n=== Korelasi Fitur Baru terhadap median_house_value ===")
print(numeric_df2.corr()['median_house_value'].sort_values(ascending=False))

# --- Visualisasi cluster lokasi di peta (longitude vs latitude) ---
plt.figure(figsize=(8, 7))
scatter = plt.scatter(df['longitude'], df['latitude'],
                       c=df['location_cluster'], cmap='tab10',
                       alpha=0.5, s=10)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Cluster Lokasi Berdasarkan Latitude & Longitude')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.savefig('cluster_lokasi.png', dpi=150)
plt.close()
print("Peta cluster lokasi disimpan sebagai 'cluster_lokasi.png'")


=== Fitur Baru yang Ditambahkan ===
   rooms_per_household  bedrooms_per_room  population_per_household  \
0             6.984127           0.146591                  2.555556   
1             6.238137           0.155797                  2.109842   
2             8.288136           0.129516                  2.802260   
3             5.817352           0.184458                  2.547945   
4             6.281853           0.172096                  2.181467   

   location_cluster  
0                 1  
1                 1  
2                 1  
3                 1  
4                 1  

=== Statistik per Cluster Lokasi (diurutkan dari harga tertinggi) ===
                  jumlah  lat_rata  long_rata  harga_rata  income_rata ocean_dominan
location_cluster                                                                    
8                   1769     37.24    -121.93   288115.47         5.01     <1H OCEAN
1                   3153     37.94    -122.32   243996.44         4.10      NE

In [9]:
# =========================================================
# SIMPAN DATASET HASIL PEMROSESAN
# =========================================================
df.to_csv('housing_processed.csv', index=False)
print("\nDataset hasil pemrosesan disimpan sebagai 'housing_processed.csv'")


Dataset hasil pemrosesan disimpan sebagai 'housing_processed.csv'
